# Carbon Intensity vs Demand Pattern Correlation

Visualizes how the demand pattern correlates with carbon intensity over time.

In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np

%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 8)

## Load Scenarios

In [ ]:
# Load carbon scenario
with open('carbon_scenario.json') as f:
    carbon_data = json.load(f)
    carbon_pattern = carbon_data['pattern']

# Load demand scenario
with open('demand_scenario.json') as f:
    demand_data = json.load(f)
    demand_pattern = demand_data['pattern']

print(f"Carbon scenario: {len(carbon_pattern)} points")
print(f"Demand scenario: {len(demand_pattern)} points")
print(f"\nCarbon range: {min(carbon_pattern)} - {max(carbon_pattern)} gCO2/kWh")
print(f"Demand range: {min(demand_pattern)} - {max(demand_pattern)}%")

## Visualize Patterns

In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(14, 12))

# Time axis (point indices)
time_points = np.arange(len(carbon_pattern))

# Plot 1: Both patterns overlaid (dual y-axis)
ax1_twin = ax1.twinx()

line1 = ax1.plot(time_points, carbon_pattern, 'r-', linewidth=2.5, label='Carbon Intensity', alpha=0.8)
line2 = ax1_twin.plot(time_points, demand_pattern, 'b--', linewidth=2.5, label='Demand', alpha=0.8)

ax1.set_xlabel('Time Point', fontsize=12)
ax1.set_ylabel('Carbon Intensity (gCO2/kWh)', fontsize=12, color='r')
ax1_twin.set_ylabel('Demand (%)', fontsize=12, color='b')
ax1.tick_params(axis='y', labelcolor='r')
ax1_twin.tick_params(axis='y', labelcolor='b')
ax1.set_title('Carbon Intensity vs Demand Pattern Over Time', fontsize=14, fontweight='bold')
ax1.grid(True, alpha=0.3)

# Combined legend
lines = line1 + line2
labels = [l.get_label() for l in lines]
ax1.legend(lines, labels, loc='upper left', fontsize=11)

# Highlight peak carbon periods
peak_carbon_threshold = 250
for i, carbon in enumerate(carbon_pattern):
    if carbon >= peak_carbon_threshold:
        ax1.axvspan(i-0.5, i+0.5, color='red', alpha=0.1)

# Plot 2: Normalized comparison (0-1 scale)
carbon_normalized = (np.array(carbon_pattern) - min(carbon_pattern)) / (max(carbon_pattern) - min(carbon_pattern))
demand_normalized = (np.array(demand_pattern) - min(demand_pattern)) / (max(demand_pattern) - min(demand_pattern))

ax2.plot(time_points, carbon_normalized, 'r-', linewidth=2.5, label='Carbon (normalized)', alpha=0.8)
ax2.plot(time_points, demand_normalized, 'b--', linewidth=2.5, label='Demand (normalized)', alpha=0.8)
ax2.fill_between(time_points, carbon_normalized, demand_normalized, alpha=0.2, color='purple')

ax2.set_xlabel('Time Point', fontsize=12)
ax2.set_ylabel('Normalized Value (0-1)', fontsize=12)
ax2.set_title('Normalized Comparison (0-1 scale)', fontsize=14, fontweight='bold')
ax2.legend(fontsize=11, loc='upper left')
ax2.grid(True, alpha=0.3)
ax2.set_ylim(-0.05, 1.05)

# Plot 3: Scatter plot showing correlation
ax3.scatter(carbon_pattern, demand_pattern, alpha=0.7, s=100, c=time_points, cmap='viridis')
ax3.set_xlabel('Carbon Intensity (gCO2/kWh)', fontsize=12)
ax3.set_ylabel('Demand (%)', fontsize=12)
ax3.set_title('Correlation: Demand vs Carbon Intensity', fontsize=14, fontweight='bold')
ax3.grid(True, alpha=0.3)

# Add correlation coefficient
correlation = np.corrcoef(carbon_pattern, demand_pattern)[0, 1]
ax3.text(0.05, 0.95, f'Correlation: {correlation:.3f}', 
         transform=ax3.transAxes, fontsize=12, verticalalignment='top',
         bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

# Add trendline
z = np.polyfit(carbon_pattern, demand_pattern, 1)
p = np.poly1d(z)
ax3.plot(carbon_pattern, p(carbon_pattern), "r--", alpha=0.8, linewidth=2, label=f'Trend: y={z[0]:.3f}x+{z[1]:.1f}')
ax3.legend(fontsize=10)

# Color bar for time progression
cbar = plt.colorbar(ax3.collections[0], ax=ax3)
cbar.set_label('Time Point', fontsize=11)

plt.tight_layout()
plt.show()

## Analysis

In [ ]:
# Calculate correlation
correlation = np.corrcoef(carbon_pattern, demand_pattern)[0, 1]

print("="*60)
print("CORRELATION ANALYSIS")
print("="*60)
print(f"\nPearson correlation coefficient: {correlation:.4f}")
print()

if correlation > 0.7:
    print("⚠️  STRONG POSITIVE CORRELATION")
    print("Demand peaks when carbon peaks!")
    print()
    print("IMPLICATIONS:")
    print("- Forecast-aware strategies that try to 'get ahead' of demand")
    print("  will serve requests during high-carbon periods")
    print("- This explains why forecast-aware-global had HIGHER emissions")
    print("  per request despite using less energy")
    print("- The demand forecasting is COUNTERPRODUCTIVE for carbon reduction")
elif correlation < -0.7:
    print("✅ STRONG NEGATIVE CORRELATION")
    print("Demand peaks when carbon is low - IDEAL for carbon-aware scheduling!")
else:
    print("ℹ️  WEAK/MODERATE CORRELATION")
    print("Demand and carbon are somewhat independent")

print()

# Find peak periods
carbon_peak_indices = [i for i, c in enumerate(carbon_pattern) if c >= 250]
demand_at_carbon_peaks = [demand_pattern[i] for i in carbon_peak_indices]

print(f"During high-carbon periods (≥250 gCO2/kWh):")
print(f"  Number of points: {len(carbon_peak_indices)}")
print(f"  Average demand: {np.mean(demand_at_carbon_peaks):.1f}%")
print(f"  Max demand: {max(demand_at_carbon_peaks):.1f}%")
print()

# Find low carbon periods
carbon_low_indices = [i for i, c in enumerate(carbon_pattern) if c <= 60]
demand_at_carbon_lows = [demand_pattern[i] for i in carbon_low_indices]

print(f"During low-carbon periods (≤60 gCO2/kWh):")
print(f"  Number of points: {len(carbon_low_indices)}")
print(f"  Average demand: {np.mean(demand_at_carbon_lows):.1f}%")
print(f"  Max demand: {max(demand_at_carbon_lows):.1f}%")
print()

## Phase Analysis

Check if demand lags behind carbon:

In [ ]:
# Calculate cross-correlation to find lag
from scipy import signal

# Normalize for cross-correlation
carbon_norm = (np.array(carbon_pattern) - np.mean(carbon_pattern)) / np.std(carbon_pattern)
demand_norm = (np.array(demand_pattern) - np.mean(demand_pattern)) / np.std(demand_pattern)

# Calculate cross-correlation
correlation_lags = signal.correlate(demand_norm, carbon_norm, mode='same')
lags = signal.correlation_lags(len(demand_norm), len(carbon_norm), mode='same')

# Find the lag with maximum correlation
max_corr_idx = np.argmax(correlation_lags)
optimal_lag = lags[max_corr_idx]

print("="*60)
print("PHASE/LAG ANALYSIS")
print("="*60)
print()
print(f"Optimal lag: {optimal_lag} time points")
print()

if optimal_lag > 0:
    print(f"⚠️  Demand LAGS carbon by {optimal_lag} points")
    print(f"    → Demand peaks AFTER carbon peaks")
    print(f"    → This is the 'demand-follows-carbon-with-lag' pattern")
elif optimal_lag < 0:
    print(f"ℹ️  Demand LEADS carbon by {abs(optimal_lag)} points")
    print(f"    → Demand peaks BEFORE carbon peaks")
else:
    print(f"ℹ️  Demand and carbon are IN PHASE")
    print(f"    → Peaks occur simultaneously")

print()
print("IMPACT ON FORECAST-AWARE STRATEGY:")
if optimal_lag >= 0:
    print("- When carbon is rising, demand is also rising")
    print("- Strategy sees high future demand during high-carbon periods")
    print("- Tries to serve NOW (during high carbon) to get ahead")
    print("- Result: MORE emissions per request")

## Save Visualization

In [ ]:
# Recreate the figure for saving
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(14, 12))

time_points = np.arange(len(carbon_pattern))

# Plot 1
ax1_twin = ax1.twinx()
line1 = ax1.plot(time_points, carbon_pattern, 'r-', linewidth=2.5, label='Carbon Intensity', alpha=0.8)
line2 = ax1_twin.plot(time_points, demand_pattern, 'b--', linewidth=2.5, label='Demand', alpha=0.8)
ax1.set_xlabel('Time Point', fontsize=12)
ax1.set_ylabel('Carbon Intensity (gCO2/kWh)', fontsize=12, color='r')
ax1_twin.set_ylabel('Demand (%)', fontsize=12, color='b')
ax1.tick_params(axis='y', labelcolor='r')
ax1_twin.tick_params(axis='y', labelcolor='b')
ax1.set_title('Carbon Intensity vs Demand Pattern Over Time', fontsize=14, fontweight='bold')
ax1.grid(True, alpha=0.3)
lines = line1 + line2
labels = [l.get_label() for l in lines]
ax1.legend(lines, labels, loc='upper left', fontsize=11)
for i, carbon in enumerate(carbon_pattern):
    if carbon >= 250:
        ax1.axvspan(i-0.5, i+0.5, color='red', alpha=0.1)

# Plot 2
ax2.plot(time_points, carbon_normalized, 'r-', linewidth=2.5, label='Carbon (normalized)', alpha=0.8)
ax2.plot(time_points, demand_normalized, 'b--', linewidth=2.5, label='Demand (normalized)', alpha=0.8)
ax2.fill_between(time_points, carbon_normalized, demand_normalized, alpha=0.2, color='purple')
ax2.set_xlabel('Time Point', fontsize=12)
ax2.set_ylabel('Normalized Value (0-1)', fontsize=12)
ax2.set_title('Normalized Comparison (0-1 scale)', fontsize=14, fontweight='bold')
ax2.legend(fontsize=11, loc='upper left')
ax2.grid(True, alpha=0.3)
ax2.set_ylim(-0.05, 1.05)

# Plot 3
ax3.scatter(carbon_pattern, demand_pattern, alpha=0.7, s=100, c=time_points, cmap='viridis')
ax3.set_xlabel('Carbon Intensity (gCO2/kWh)', fontsize=12)
ax3.set_ylabel('Demand (%)', fontsize=12)
ax3.set_title('Correlation: Demand vs Carbon Intensity', fontsize=14, fontweight='bold')
ax3.grid(True, alpha=0.3)
correlation = np.corrcoef(carbon_pattern, demand_pattern)[0, 1]
ax3.text(0.05, 0.95, f'Correlation: {correlation:.3f}', 
         transform=ax3.transAxes, fontsize=12, verticalalignment='top',
         bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
z = np.polyfit(carbon_pattern, demand_pattern, 1)
p = np.poly1d(z)
ax3.plot(carbon_pattern, p(carbon_pattern), "r--", alpha=0.8, linewidth=2, label=f'Trend: y={z[0]:.3f}x+{z[1]:.1f}')
ax3.legend(fontsize=10)
cbar = plt.colorbar(ax3.collections[0], ax=ax3)
cbar.set_label('Time Point', fontsize=11)

plt.tight_layout()
plt.savefig('carbon_demand_correlation.png', dpi=150, bbox_inches='tight')
print("📊 Saved to: carbon_demand_correlation.png")
plt.show()